# 01 – Data tour: what the ingestion pipeline does

In [ ]:
import json
import os
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
os.chdir(ROOT)
print("Working from:", ROOT)

pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.figsize": (11, 3.5), "axes.grid": True, "grid.alpha": 0.3})

### 1.1 What the NESO API returns
`demand.py` first asks NESO's CKAN API to list every file in the dataset. `select_years` then filters it to one CSV URL per year.

In [ ]:
import httpx

from gb_demand_forecast.ingest.demand import CKAN_BASE, DATASET_ID, select_years

try:
    response = httpx.get(
        f"{CKAN_BASE}/package_show", params={"id": DATASET_ID}, timeout=30, follow_redirects=True
    )
    resources = response.json()["result"]["resources"]
    display(pd.DataFrame(resources)[["name", "format", "url"]].head(12))
    print("select_years keeps:", sorted(select_years(resources, 2018, date.today().year)))
except httpx.HTTPError as err:
    print("Skipped - no connection to NESO:", err)

### 1.2 The files it saved
One untouched CSV per year. `download()` skips past years if they already exist and always refreshes the current one.

In [ ]:
raw_dir = Path("data/raw/demand")
files = sorted(raw_dir.glob("*.csv"))
pd.DataFrame([{"file": p.name, "size_MB": round(p.stat().st_size / 1e6, 2)} for p in files])

### 1.3 How the yearly files differ
This is why `process_demand.py` exists: the files aren't consistent. Look at
`first_date` (three date formats) and `max_period` (50 = a clocks-back day; 2026's October
change hasn't happened yet).

In [ ]:
raw = {p.stem[-4:]: pd.read_csv(p) for p in sorted(raw_dir.glob("demand_*.csv"))}

pd.DataFrame(
    {
        year: {
            "rows": len(df),
            "columns": df.shape[1],
            "first_date": df["SETTLEMENT_DATE"].iloc[0],
            "last_date": df["SETTLEMENT_DATE"].iloc[-1],
            "max_period": df["SETTLEMENT_PERIOD"].max(),
        }
        for year, df in raw.items()
    }
).T

Which columns appear in which year. `COLUMNS` in `process_demand.py` picks only the six we need, so the changing extras don't matter.

In [ ]:
all_columns = sorted(set().union(*[df.columns for df in raw.values()]))
presence = pd.DataFrame(
    {year: [col in df.columns for col in all_columns] for year, df in raw.items()},
    index=all_columns,
)
presence.replace({True: "Y", False: ""})

In [ ]:
# A raw file exactly as NESO publishes it
raw["2018"].head()

---
## Part 2 – What `process_demand.py` does

Compare a guessing parser with the explicit-format parser on a
few ISO dates. Days 1–12 get their day and month swapped; day 13+ can't be swapped.

In [ ]:
from gb_demand_forecast.ingest.process_demand import parse_settlement_dates

sample = pd.Series(["2025-02-05", "2025-01-10", "2025-12-25", "2025-07-20"])
pd.DataFrame(
    {
        "raw text": sample,
        "guessing parser": pd.to_datetime(sample, format="mixed", dayfirst=True).dt.strftime(
            "%d %b %Y"
        ),
        "explicit formats": parse_settlement_dates(sample).dt.strftime("%d %b %Y"),
    }
)

### 2.2 Settlement periods → UTC on a clock-change day
Period 1 starts at UK midnight and each period is 30 real minutes. On 26 October 2025 the
clocks went back, so the day had **50** periods. Look at `uk_clock_time`: 01:00–01:30
happens **twice** (BST then GMT), but `timestamp_utc` never repeats. That's why we store
UTC.

In [ ]:
from gb_demand_forecast.timeutils import expected_periods, settlement_period_to_utc

for day in ["2025-06-15", "2025-03-30", "2025-10-26"]:
    print(day, "->", expected_periods(day), "periods")

day = "2025-10-26"
n = expected_periods(day)
demo = pd.DataFrame({"settlement_period": range(1, n + 1)})
demo["timestamp_utc"] = settlement_period_to_utc(pd.Series([day] * n), demo["settlement_period"])
demo["uk_clock_time"] = demo["timestamp_utc"].dt.tz_convert("Europe/London")
demo.iloc[[0, 1, 2, 3, 4, 5, 6, -2, -1]]

### 2.3 One raw file after `load_raw_year`
Columns renamed, only six kept, forecasts dropped (if any), dates parsed. Timestamps are added later in `build_demand_table`.

In [ ]:
from gb_demand_forecast.ingest.process_demand import load_raw_year

load_raw_year(raw_dir / "demand_2023.csv").head()

---
## Part 3 – The processed demand table

One row per half-hour, 2018 onwards, with a timezone-aware UTC timestamp.

In [ ]:
demand = pd.read_parquet("data/processed/demand.parquet")
demand.info()
demand.head()

In [ ]:
ts = demand["timestamp_utc"]
print("Range:", ts.min(), "to", ts.max())
print("Unique timestamps:", ts.is_unique)
print("Gaps > 30 min:", (ts.diff() > pd.Timedelta(minutes=30)).sum())

### 3.1 One week of demand

In [ ]:
nd = demand.set_index("timestamp_utc")["nd"]
week = nd.loc["2025-01-20":"2025-01-26"]
week.index = week.index.tz_convert("Europe/London")
ax = week.plot(title="National Demand, week of 20 Jan 2025 (UK time)")
ax.set_ylabel("MW");

### 3.2 The long view
Daily average with a 30-day rolling mean. Winter peaks, summer troughs, and a slow decline as embedded solar and efficiency reduce demand on the transmission network.

In [ ]:
daily_nd = nd.resample("D").mean()
ax = daily_nd.plot(alpha=0.3, label="daily mean")
daily_nd.rolling(30, center=True).mean().plot(ax=ax, label="30-day rolling mean")
ax.set_ylabel("MW")
ax.legend();

### 3.3 The average day, winter vs summer
Grouped by UK clock time. Summer middays dip because embedded solar supplies homes directly, so less demand reaches the national grid.

In [ ]:
local = demand.assign(uk=demand["timestamp_utc"].dt.tz_convert("Europe/London"))
local["time_of_day"] = local["uk"].dt.hour + local["uk"].dt.minute / 60
seasons = {12: "winter (Dec-Feb)", 1: "winter (Dec-Feb)", 2: "winter (Dec-Feb)"}
seasons |= {6: "summer (Jun-Aug)", 7: "summer (Jun-Aug)", 8: "summer (Jun-Aug)"}
local["season"] = local["uk"].dt.month.map(seasons)  # other months become NaN
profile = local.dropna(subset=["season"]).pivot_table(
    index="time_of_day", columns="season", values="nd", aggfunc="mean"
)
ax = profile.plot(title="Average demand by time of day")
ax.set_xlabel("UK clock hour")
ax.set_ylabel("MW");

---
## Part 4 – Raw weather

### 4.1 Where and what we ask for
Eight cities, weighted by population. `build_params` builds the API request; note the
`_previous_day2` suffix (forecasts issued 48 hours ahead) and the pinned weather model.

In [ ]:
from gb_demand_forecast.ingest.weather import CITIES, build_params

cities = pd.DataFrame(CITIES, index=["lat", "lon", "population_m"]).T
cities["weight_%"] = (100 * cities["population_m"] / cities["population_m"].sum()).round(1)
cities

In [ ]:
build_params("forecast", *CITIES["london"][:2], date(2025, 1, 1), date(2025, 12, 31))

### 4.2 One raw response
Saved exactly as returned. Note the coordinates snapped to the model's 0.25° grid, and the hourly arrays.

In [ ]:
payload = json.loads(Path("data/raw/weather/forecast/london_2025.json").read_text())
print({k: v for k, v in payload.items() if k != "hourly"})
pd.DataFrame(payload["hourly"]).head()

---
## Part 5 – What `process_weather.py` does

### 5.1 Population-weighted GB average
Check one hour by hand: multiply each city by its weight, sum, divide by total weight.
It should match what `weighted_average` produced.

In [ ]:
from gb_demand_forecast.ingest.process_weather import load_city, to_half_hourly, weighted_average

frames = {city: load_city("forecast", city) for city in CITIES}
weights = {city: pop for city, (_, _, pop) in CITIES.items()}
gb_hourly = weighted_average(frames, weights).dropna(how="all")

t = pd.Timestamp("2025-01-15 12:00", tz="UTC")
one_hour = pd.DataFrame({city: frame.loc[t] for city, frame in frames.items()}).T
one_hour["weight"] = pd.Series(weights)
by_hand = (one_hour["temperature_2m"] * one_hour["weight"]).sum() / one_hour["weight"].sum()
print(f"By hand: {by_hand:.3f}   weighted_average: {gb_hourly.loc[t, 'temperature_2m']:.3f}")
one_hour

### 5.2 Hourly → half-hourly
Temperature is a value *at* the hour, so 00:30 is interpolated between 00:00 and 01:00.
Radiation is an *average over the previous hour* labelled at the hour's end, so it's
shifted back 30 minutes first. In the right-hand plot the half-hourly line sits half an
hour earlier than the raw hourly points: that's the re-centring.

In [ ]:
gb_half_hourly = to_half_hourly(gb_hourly)
day = "2025-06-21"
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, col in zip(axes, ["temperature_2m", "shortwave_radiation"], strict=True):
    gb_hourly.loc[day, col].plot(ax=ax, marker="o", linestyle="", label="hourly (raw)")
    gb_half_hourly.loc[day, col].plot(ax=ax, marker=".", label="half-hourly (processed)")
    ax.set_title(col)
    ax.legend()

---
## Part 6 – The processed weather tables

### 6.1 Coverage and gaps
Observed runs from 2018; leak-free forecasts from Feb 2024 (radiation from Mar 2024), with a
known cloud-cover gap in April 2026.

In [ ]:
obs = pd.read_parquet("data/processed/weather_observed.parquet").set_index("timestamp_utc")
fc = pd.read_parquet("data/processed/weather_forecast.parquet").set_index("timestamp_utc")

pd.DataFrame(
    {
        "observed: first": obs.apply(lambda s: s.first_valid_index()),
        "forecast: first": fc.apply(lambda s: s.first_valid_index()),
        "forecast: missing": fc.isna().sum(),
    }
)

### 6.2 How good is a 2-day-ahead forecast?
Forecast vs what actually happened, over the overlap.

In [ ]:
both = obs.join(fc, lsuffix="_obs", rsuffix="_fc", how="inner").dropna()
error = both["temperature_2m_fc"] - both["temperature_2m_obs"]
print(f"Temperature MAE: {error.abs().mean():.2f} C   bias: {error.mean():+.2f} C")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(both["temperature_2m_obs"], both["temperature_2m_fc"], s=1, alpha=0.1)
axes[0].axline((0, 0), slope=1, color="black", linewidth=1)
axes[0].set(xlabel="observed (C)", ylabel="day-2 forecast (C)", title="Forecast vs observed")
error.hist(bins=60, ax=axes[1])
axes[1].set(xlabel="forecast - observed (C)", title="Forecast error");

---
## Part 7 – First look at the relationship we care about

Daily average demand against daily average temperature.

In [ ]:
daily = nd.to_frame().join(obs[["temperature_2m"]], how="inner").resample("D").mean().dropna()
weekend = daily.index.dayofweek >= 5
ax = daily[~weekend].plot.scatter(x="temperature_2m", y="nd", s=3, alpha=0.4, label="weekday")
daily[weekend].plot.scatter(
    x="temperature_2m", y="nd", s=3, alpha=0.4, color="tab:orange", ax=ax, label="weekend"
)
ax.set(xlabel="daily mean temperature (C)", ylabel="daily mean demand (MW)");